# 🧬 LLPS Protein Data Explorer - Step-by-Step Analysis

This notebook provides a transparent, step-by-step view of all data transformations applied to the LLPS protein dataset. Each cell shows exactly what is happening to your data at each stage.

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Initial Data Inspection](#2-initial-data-inspection)
3. [Location Parsing Transformations](#3-location-parsing-transformations)
4. [Data Filtering Examples](#4-data-filtering-examples)
5. [Statistical Analysis](#5-statistical-analysis)
6. [Visualizations](#6-visualizations)

---
## 1. Setup & Data Loading

First, we import all necessary libraries and load the data.

In [1]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import re

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Load the sample data
data_path = Path("/workspaces/mem_prot_llps/Human Phase separation data.xlsx")

if data_path.exists():
    df_raw = pd.read_excel(data_path, engine='openpyxl')
    print(f"✅ Data loaded successfully!")
    print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
else:
    print("❌ Sample data not found. Please ensure data/sample_data.xlsx exists.")
    print("   You can also upload your own XLSX file and update the path above.")

✅ Data loaded successfully!
   Shape: 20366 rows × 11 columns


---
## 2. Initial Data Inspection

Let's examine the raw data before any transformations.

In [3]:
# View all column names
print("📋 Column names in the dataset:")
print("=" * 50)
for i, col in enumerate(df_raw.columns, 1):
    print(f"{i:2}. {col}")

📋 Column names in the dataset:
 1. Entry
 2. Entry name
 3. Protein names
 4. p(LLPS)
 5. n(DPR=> 25)
 6. Organism
 7. Length
 8. Function [CC]
 9. Subcellular location [CC]
10. Involvement in disease
11. Cross-reference (PDB)


In [4]:
# Data types and non-null counts
print("📊 Data Types and Non-Null Counts:")
print("=" * 50)
df_raw.info()

📊 Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20366 entries, 0 to 20365
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Entry                      20366 non-null  object 
 1   Entry name                 20366 non-null  object 
 2   Protein names              20366 non-null  object 
 3   p(LLPS)                    20366 non-null  float64
 4   n(DPR=> 25)                20366 non-null  int64  
 5   Organism                   20363 non-null  object 
 6   Length                     20363 non-null  float64
 7   Function [CC]              16469 non-null  object 
 8   Subcellular location [CC]  16692 non-null  object 
 9   Involvement in disease     4454 non-null   object 
 10  Cross-reference (PDB)      6890 non-null   object 
dtypes: float64(2), int64(1), object(8)
memory usage: 1.7+ MB


In [5]:
# Preview the first few rows
print("👀 First 5 rows of raw data:")
df_raw.head()

👀 First 5 rows of raw data:


,Entry,Entry name,Protein names,p(LLPS),n(DPR=> 25),Organism,Length,Function [CC],Subcellular location [CC],Involvement in disease,Cross-reference (PDB)
0,Q9Y6V0,PCLO_HUMAN,Protein piccolo (Aczonin),1.0,21,Homo sapiens (Human),5142.0,Scaffold protein of the presynaptic cytomatri...,"Cell junction, synapse, presynaptic active zo...",Pontocerebellar hypoplasia 3 (PCH3) [MIM:6080...,1UJD;
1,Q9Y566,SHAN1_HUMAN,SH3 and multiple ankyrin repeat domains protei...,1.0,8,Homo sapiens (Human),2161.0,Seems to be an adapter protein in the postsyn...,"Cytoplasm {ECO:0000250}. Cell junction, synap...",NaN,6CPI;
2,Q9Y520,PRC2C_HUMAN,Protein PRRC2C (BAT2 domain-containing protein...,1.0,17,Homo sapiens (Human),2896.0,Required for efficient formation of stress gr...,"Cytoplasm, Stress granule {ECO:0000305|PubMed...",NaN,NaN
3,Q9Y4H2,IRS2_HUMAN,Insulin receptor substrate 2 (IRS-2),1.0,9,Homo sapiens (Human),1338.0,May mediate the control of various cellular p...,"Cytoplasm, cytosol {ECO:0000250}.",NaN,3FQW;3FQX;
4,Q9Y3S1,WNK2_HUMAN,Serine/threonine-protein kinase WNK2 (EC 2.7.1...,1.0,18,Homo sapiens (Human),2297.0,Serine/threonine kinase which plays an import...,"Cytoplasm {ECO:0000269|PubMed:17667937, ECO:0...",NaN,6ELM;6FBK;


In [6]:
# Basic statistics for numeric columns
print("📈 Statistical summary of numeric columns:")
df_raw.describe()

📈 Statistical summary of numeric columns:


,p(LLPS),n(DPR=> 25),Length
count,20366.000000,20366.000000,20363.000000
mean,0.487834,1.649563,557.740755
std,0.336152,2.547905,596.870676
min,0.060000,0.000000,2.000000
25%,0.180000,0.000000,250.000000
50%,0.360000,1.000000,415.000000
75%,0.860000,2.000000,670.000000
max,1.000000,110.000000,34350.000000


In [7]:
# Check for missing values
print("🔍 Missing values per column:")
print("=" * 50)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False))
print(f"\nTotal rows: {len(df_raw)}")

🔍 Missing values per column:
                           Missing Count  Missing %
Involvement in disease             15912       78.1
Cross-reference (PDB)              13476       66.2
Function [CC]                       3897       19.1
Subcellular location [CC]           3674       18.0
Length                                 3        0.0
Organism                               3        0.0

Total rows: 20366


---
## 3. Location Parsing Transformations

The dashboard parses subcellular location strings into standardized categories. Let's see this step-by-step.

In [8]:
# No predefined categories needed - we extract location tags directly
# from the UniProt annotation, removing only the evidence codes in curly brackets
print("ℹ️ Location parsing approach:")
print("   1. Remove curly bracket content like {ECO:xxx}")
print("   2. Split by separators (comma, semicolon, period)")
print("   3. Extract unique location tags for analysis")
print("\n💡 The original column preserves full annotation details")

ℹ️ Location parsing approach:
   1. Remove curly bracket content like {ECO:xxx}
   2. Split by separators (comma, semicolon, period)
   3. Extract unique location tags for analysis

💡 The original column preserves full annotation details


In [9]:
# Define the location parsing function
def parse_location(location_str):
    """
    Parse a subcellular location string from UniProt and return a list of location terms.
    
    Process:
    1. Remove curly bracket annotations like {ECO:xxx} for cleaner output
    2. Remove normal bracket annotations like (By similarity) for cleaner output
    3. Remove square bracket content like [Isoform 1]: or [Note: ...]
    4. Remove Isoform tags without brackets like Isoform 1:
    5. Remove everything after 'Note=' for cleaner output
    6. Split by common separators (comma, semicolon, period)
    7. Extract unique location tags for analysis and plotting
    
    The original column can be referenced for full annotation details if needed.
    """
    if pd.isna(location_str) or location_str == '':
        return []
    
    location_str = str(location_str)
    
    # Remove curly bracket content like {ECO:0000269|PubMed:12345}
    location_str = re.sub(r'\{[^}]*\}', '', location_str)
    
    # Remove all square bracket content, optionally followed by colon
    # This handles [Isoform 1]: and [Note: ...]
    location_str = re.sub(r'\[[^\]]*\]:?', '', location_str)
    
    # Remove Isoform ...: tags (without brackets)
    location_str = re.sub(r'Isoform\s+[^:]+:\s*', '', location_str, flags=re.IGNORECASE)
    
    # Remove normal bracket content like (By similarity) or (Potential)
    location_str = re.sub(r'\([^)]*\)', '', location_str)
    
    # Remove "SUBCELLULAR LOCATION:" prefix if present
    location_str = re.sub(r'^SUBCELLULAR LOCATION:\s*', '', location_str, flags=re.IGNORECASE)
    
    # Remove everything after 'Note=' (case-insensitive)
    location_str = re.sub(r'\s*Note=.*', '', location_str, flags=re.IGNORECASE)
    
    # Split by common separators and clean up
    # UniProt uses semicolons, commas, and periods as separators
    parts = re.split(r'[;,.]', location_str)
    
    locations = []
    for part in parts:
        # Clean up whitespace and filter empty/very short strings
        cleaned = part.strip()
        if len(cleaned) < 2:  # Skip empty or single-char remnants
            continue
        if cleaned not in locations:
            locations.append(cleaned)
    
    return locations

print("✅ Location parsing function defined")

✅ Location parsing function defined


In [10]:
# Demonstrate the parsing on a few examples
if 'Subcellular location [CC]' in df_raw.columns:
    print("🔬 Location parsing examples:")
    print("=" * 80)
    
    # Get a sample of unique location strings
    sample_locations = df_raw['Subcellular location [CC]'].dropna().unique()[:5]
    
    for loc in sample_locations:
        parsed = parse_location(loc)
        print(f"\nOriginal: {str(loc)[:70]}..." if len(str(loc)) > 70 else f"\nOriginal: {loc}")
        print(f"  → Parsed locations: {parsed}")
else:
    print("ℹ️ Column 'Subcellular location [CC]' not found in data.")

🔬 Location parsing examples:

Original:  Cell junction, synapse, presynaptic active zone {ECO:0000250|UniProtK...
  → Parsed locations: ['Cell junction', 'synapse', 'presynaptic active zone']

Original:  Cytoplasm {ECO:0000250}. Cell junction, synapse, postsynaptic density...
  → Parsed locations: ['Cytoplasm', 'Cell junction', 'synapse', 'postsynaptic density']

Original:  Cytoplasm, Stress granule {ECO:0000305|PubMed:29395067}.
  → Parsed locations: ['Cytoplasm', 'Stress granule']

Original:  Cytoplasm, cytosol {ECO:0000250}.
  → Parsed locations: ['Cytoplasm', 'cytosol']

Original:  Cytoplasm {ECO:0000269|PubMed:17667937, ECO:0000269|PubMed:18593598}....
  → Parsed locations: ['Cytoplasm', 'Cell membrane']


In [11]:
# Apply the transformations to create a working copy
print("🔄 Applying location parsing transformations...")
print("=" * 50)

df = df_raw.copy()
print(f"Step 1: Created working copy of dataframe")
print(f"        Shape: {df.shape}")

if 'Subcellular location [CC]' in df.columns:
    # Add Location Categories column (list of all parsed locations)
    df['Location Categories'] = df['Subcellular location [CC]'].apply(parse_location)
    print(f"\nStep 2: Added 'Location Categories' column")
    print(f"        Example: {df['Location Categories'].iloc[0]}")
    
    # Show all unique locations found
    all_locs = set()
    for locs in df['Location Categories']:
        all_locs.update(locs)
    print(f"\n        All unique locations found: {sorted(all_locs)}")
else:
    print("\nSkipping location parsing - column not found")

print(f"\n✅ Final shape after transformations: {df.shape}")
print(f"   New columns added: {set(df.columns) - set(df_raw.columns)}")

🔄 Applying location parsing transformations...
Step 1: Created working copy of dataframe
        Shape: (20366, 11)

Step 2: Added 'Location Categories' column
        Example: ['Cell junction', 'synapse', 'presynaptic active zone']

        All unique locations found: ['A band', 'Apical cell membrane', 'Apicolateral cell membrane', 'Basal cell membrane', 'Basolateral cell membrane', 'COPI-coated vesicle', 'COPI-coated vesicle membrane', 'COPII-coated vesicle', 'COPII-coated vesicle membrane', 'Cajal body', 'Cell junction', 'Cell membrane', 'Cell projection', 'Cell surface', 'Chlorosome', 'Chromosome', 'Cleavage furrow', 'Cornified envelope', 'Cytoplasm', 'Cytoplasmic granule', 'Cytoplasmic granule lumen', 'Cytoplasmic granule membrane', 'Cytoplasmic ribonucleoprotein granule', 'Cytoplasmic side', 'Cytoplasmic vesicle', 'Cytoplasmic vesicle lumen', 'Cytoplasmic vesicle membrane', 'Early endosome', 'Early endosome membrane', 'Endomembrane system', 'Endoplasmic reticulum', 'Endoplasmic r

In [12]:
# View the transformed data
print("👀 Preview of transformed data (showing location columns):")
cols_to_show = ['Entry', 'Entry name']
if 'Subcellular location [CC]' in df.columns:
    cols_to_show.append('Subcellular location [CC]')
if 'Location Categories' in df.columns:
    cols_to_show.append('Location Categories')

# Filter to columns that exist
cols_to_show = [c for c in cols_to_show if c in df.columns]
df[cols_to_show].head(10)

👀 Preview of transformed data (showing location columns):


,Entry,Entry name,Subcellular location [CC],Location Categories
0,Q9Y6V0,PCLO_HUMAN,"Cell junction, synapse, presynaptic active zo...","[Cell junction, synapse, presynaptic active zone]"
1,Q9Y566,SHAN1_HUMAN,"Cytoplasm {ECO:0000250}. Cell junction, synap...","[Cytoplasm, Cell junction, synapse, postsynapt..."
2,Q9Y520,PRC2C_HUMAN,"Cytoplasm, Stress granule {ECO:0000305|PubMed...","[Cytoplasm, Stress granule]"
3,Q9Y4H2,IRS2_HUMAN,"Cytoplasm, cytosol {ECO:0000250}.","[Cytoplasm, cytosol]"
4,Q9Y3S1,WNK2_HUMAN,"Cytoplasm {ECO:0000269|PubMed:17667937, ECO:0...","[Cytoplasm, Cell membrane]"
5,Q9UPQ9,TNR6B_HUMAN,"Cytoplasm, P-body {ECO:0000269|PubMed:1628964...","[Cytoplasm, P-body]"
6,Q9UPA5,BSN_HUMAN,Cytoplasm {ECO:0000250|UniProtKB:O88778}. Cel...,"[Cytoplasm, Cell junction, synapse, presynapti..."
7,Q9UMD9,COHA1_HUMAN,"Cell junction, hemidesmosome. Membrane; Singl...","[Cell junction, hemidesmosome, Membrane, Singl..."
8,Q9ULL5,PRR12_HUMAN,Nucleus {ECO:0000250|UniProtKB:E9PYL2}. Cell ...,"[Nucleus, Cell junction, synapse, postsynaptic..."
9,Q9ULI4,KI26A_HUMAN,"Cytoplasm, cytoskeleton {ECO:0000305}.","[Cytoplasm, cytoskeleton]"


---
## 4. Data Filtering Examples

Here we demonstrate the various filtering operations available in the dashboard.

In [13]:
# Filter by p(LLPS) range
if 'p(LLPS)' in df.columns:
    print("🎚️ Filtering by p(LLPS) range:")
    print("=" * 50)
    
    pllps_min = df['p(LLPS)'].min()
    pllps_max = df['p(LLPS)'].max()
    print(f"Full range: {pllps_min:.3f} to {pllps_max:.3f}")
    
    # Example: Filter for high probability LLPS proteins
    threshold = 0.7
    df_high_pllps = df[df['p(LLPS)'] >= threshold]
    print(f"\nFilter: p(LLPS) >= {threshold}")
    print(f"Result: {len(df_high_pllps)} proteins (from {len(df)} total)")
    print(f"\nTop 5 high p(LLPS) proteins:")
    display_cols = ['Entry', 'Entry name', 'p(LLPS)']
    display_cols = [c for c in display_cols if c in df.columns]
    print(df_high_pllps.nlargest(5, 'p(LLPS)')[display_cols])
else:
    print("ℹ️ Column 'p(LLPS)' not found in data.")

🎚️ Filtering by p(LLPS) range:
Full range: 0.060 to 1.000

Filter: p(LLPS) >= 0.7
Result: 6657 proteins (from 20366 total)

Top 5 high p(LLPS) proteins:
    Entry   Entry name  p(LLPS)
0  Q9Y6V0   PCLO_HUMAN      1.0
1  Q9Y566  SHAN1_HUMAN      1.0
2  Q9Y520  PRC2C_HUMAN      1.0
3  Q9Y4H2   IRS2_HUMAN      1.0
4  Q9Y3S1   WNK2_HUMAN      1.0


In [14]:
# Filter by protein length
if 'Length' in df.columns:
    print("📏 Filtering by protein length:")
    print("=" * 50)
    
    length_min = df['Length'].min()
    length_max = df['Length'].max()
    print(f"Full range: {length_min} to {length_max} amino acids")
    
    # Example: Filter for short proteins
    max_length = 300
    df_short = df[df['Length'] <= max_length]
    print(f"\nFilter: Length <= {max_length}")
    print(f"Result: {len(df_short)} proteins (from {len(df)} total)")
else:
    print("ℹ️ Column 'Length' not found in data.")

📏 Filtering by protein length:
Full range: 2.0 to 34350.0 amino acids

Filter: Length <= 300
Result: 6474 proteins (from 20366 total)


In [15]:
# Filter by subcellular location
if 'Location Categories' in df.columns:
    print("📍 Filtering by subcellular location:")
    print("=" * 50)
    
    # Count proteins by location (proteins can have multiple locations)
    from collections import Counter
    location_counter = Counter()
    for locs in df['Location Categories']:
        location_counter.update(locs)
    
    print("Available locations and counts:")
    for loc, count in location_counter.most_common():
        print(f"  {loc}: {count}")
    
    # Example: Filter for nuclear proteins
    location_filter = 'Nucleus'
    df_nuclear = df[df['Location Categories'].apply(lambda x: location_filter in x)]
    print(f"\nFilter: '{location_filter}' in Location Categories")
    print(f"Result: {len(df_nuclear)} proteins")
    
    if 'p(LLPS)' in df.columns and len(df_nuclear) > 0:
        print(f"Mean p(LLPS) for nuclear proteins: {df_nuclear['p(LLPS)'].mean():.3f}")
else:
    print("ℹ️ Location columns not available.")

📍 Filtering by subcellular location:
Available locations and counts:
  Cytoplasm: 5183
  Nucleus: 5137
  Cell membrane: 3429
  Multi-pass membrane protein: 2815
  Membrane: 2038
  Secreted: 2002
  cytoskeleton: 1253
  Single-pass type I membrane protein: 1142
  Peripheral membrane protein: 983
  Cell projection: 880
  Endoplasmic reticulum membrane: 809
  Cell junction: 732
  Single-pass membrane protein: 665
  Mitochondrion: 565
  Golgi apparatus: 499
  cytosol: 487
  Cytoplasmic vesicle: 475
  Lipid-anchor: 470
  Cytoplasmic side: 470
  Chromosome: 446
  microtubule organizing center: 433
  centrosome: 425
  Single-pass type II membrane protein: 423
  Golgi apparatus membrane: 407
  synapse: 405
  nucleolus: 403
  extracellular space: 333
  Mitochondrion inner membrane: 301
  Endoplasmic reticulum: 289
  perinuclear region: 272
  extracellular matrix: 262
  cilium: 242
  nucleoplasm: 191
  Nucleus speckle: 190
  Lysosome membrane: 187
  Mitochondrion matrix: 185
  spindle: 176
  dend

In [16]:
# Combined filtering
print("🔗 Combined filtering example:")
print("=" * 50)

df_filtered = df.copy()
filters_applied = []

# Apply p(LLPS) filter
if 'p(LLPS)' in df.columns:
    pllps_threshold = 0.5
    df_filtered = df_filtered[df_filtered['p(LLPS)'] >= pllps_threshold]
    filters_applied.append(f"p(LLPS) >= {pllps_threshold}")

# Apply length filter
if 'Length' in df.columns:
    min_length, max_length = 100, 1000
    df_filtered = df_filtered[(df_filtered['Length'] >= min_length) & 
                               (df_filtered['Length'] <= max_length)]
    filters_applied.append(f"Length between {min_length} and {max_length}")

print("Filters applied:")
for f in filters_applied:
    print(f"  • {f}")

print(f"\nOriginal count: {len(df)}")
print(f"Filtered count: {len(df_filtered)}")
print(f"Removed: {len(df) - len(df_filtered)} proteins")

🔗 Combined filtering example:
Filters applied:
  • p(LLPS) >= 0.5
  • Length between 100 and 1000

Original count: 20366
Filtered count: 6580
Removed: 13786 proteins


In [17]:
# Text search example
print("🔍 Text search example:")
print("=" * 50)

search_term = "kinase"  # Example search term
search_columns = [col for col in ['Entry', 'Entry name', 'Protein names'] if col in df.columns]

if search_columns:
    mask = pd.Series([False] * len(df))
    for col in search_columns:
        mask = mask | df[col].astype(str).str.contains(search_term, case=False, na=False)
    
    df_search = df[mask]
    print(f"Search term: '{search_term}'")
    print(f"Searched in columns: {search_columns}")
    print(f"\nFound {len(df_search)} matching proteins:")
    
    if len(df_search) > 0:
        print(df_search[search_columns].head(10))
else:
    print("No searchable columns found.")

🔍 Text search example:
Search term: 'kinase'
Searched in columns: ['Entry', 'Entry name', 'Protein names']

Found 881 matching proteins:
      Entry   Entry name                                      Protein names
4    Q9Y3S1   WNK2_HUMAN  Serine/threonine-protein kinase WNK2 (EC 2.7.1...
14   Q9H4A3   WNK1_HUMAN  Serine/threonine-protein kinase WNK1 (EC 2.7.1...
50   Q12802  AKP13_HUMAN  A-kinase anchor protein 13 (AKAP-13) (AKAP-Lbc...
89   O15021  MAST4_HUMAN  Microtubule-associated serine/threonine-protei...
96   Q96L96  ALPK3_HUMAN  Alpha-protein kinase 3 (EC 2.7.11.1) (Muscle a...
126  Q14004  CDK13_HUMAN  Cyclin-dependent kinase 13 (EC 2.7.11.22) (EC ...
152  Q02952  AKA12_HUMAN  A-kinase anchor protein 12 (AKAP-12) (A-kinase...
158  Q86TB3  ALPK2_HUMAN  Alpha-protein kinase 2 (EC 2.7.11.1) (Heart al...
171  Q6P0Q8  MAST2_HUMAN  Microtubule-associated serine/threonine-protei...
195  Q9NYV4  CDK12_HUMAN  Cyclin-dependent kinase 12 (EC 2.7.11.22) (EC ...


---
## 5. Statistical Analysis

Detailed statistical analysis of the dataset.

In [18]:
# Summary statistics by location
if 'Location Categories' in df.columns and 'p(LLPS)' in df.columns:
    print("📊 Statistics by Subcellular Location:")
    print("=" * 70)
    
    # Explode location categories for analysis
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    
    location_stats = df_exploded.groupby('Location Categories').agg({
        'Entry': 'count',
        'p(LLPS)': ['mean', 'std', 'min', 'max']
    }).round(3)
    
    location_stats.columns = ['Count', 'Mean p(LLPS)', 'Std p(LLPS)', 'Min p(LLPS)', 'Max p(LLPS)']
    location_stats = location_stats.sort_values('Count', ascending=False)
    
    print(location_stats)
    print("\n💡 Note: Proteins with multiple locations are counted in each location")
else:
    print("ℹ️ Required columns for location statistics not available.")

📊 Statistics by Subcellular Location:
                             Count  Mean p(LLPS)  Std p(LLPS)  Min p(LLPS)  \
Location Categories                                                          
Cytoplasm                     5183         0.536        0.338         0.06   
Nucleus                       5137         0.638        0.332         0.06   
Cell membrane                 3429         0.372        0.292         0.06   
Multi-pass membrane protein   2815         0.272        0.235         0.06   
Membrane                      2038         0.437        0.317         0.06   
...                            ...           ...          ...          ...   
microneme membrane               1         0.440          NaN         0.44   
nucleolus fibrillar center       1         0.510          NaN         0.51   
multivesicular body lumen        1         0.240          NaN         0.24   
pseudopodium membrane            1         0.170          NaN         0.17   
spindle pole body         

In [ ]:
# Correlation analysis
print("📈 Correlation Analysis:")
print("=" * 50)

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

if len(numeric_cols) >= 2:
    corr_matrix = df[numeric_cols].corr()
    print("\nCorrelation matrix:")
    print(corr_matrix.round(3))
    
    # Highlight specific correlations
    if 'Length' in numeric_cols and 'p(LLPS)' in numeric_cols:
        corr_length_pllps = df['Length'].corr(df['p(LLPS)'])
        print(f"\n🔗 Length vs p(LLPS) correlation: {corr_length_pllps:.4f}")
else:
    print("Not enough numeric columns for correlation analysis.")

In [ ]:
# Distribution percentiles
if 'p(LLPS)' in df.columns:
    print("📊 p(LLPS) Distribution Percentiles:")
    print("=" * 50)
    
    percentiles = [10, 25, 50, 75, 90, 95, 99]
    for p in percentiles:
        value = df['p(LLPS)'].quantile(p/100)
        count_above = (df['p(LLPS)'] >= value).sum()
        print(f"  {p:2}th percentile: {value:.3f} ({count_above} proteins at or above)")

---
## 6. Visualizations

Interactive visualizations to explore the data.

In [ ]:
# p(LLPS) distribution histogram
if 'p(LLPS)' in df.columns:
    print("📊 Distribution of p(LLPS):")
    
    fig = px.histogram(
        df, 
        x='p(LLPS)',
        nbins=50,
        title="Distribution of LLPS Probability",
        labels={'p(LLPS)': 'Probability of LLPS'},
        color_discrete_sequence=['#1f77b4']
    )
    fig.update_layout(
        xaxis_title="p(LLPS)",
        yaxis_title="Count",
        showlegend=False
    )
    fig.show()
else:
    print("ℹ️ Column 'p(LLPS)' not found.")

In [ ]:
# Protein length distribution
if 'Length' in df.columns:
    print("📏 Distribution of Protein Lengths:")
    
    fig = px.histogram(
        df,
        x='Length',
        nbins=50,
        title="Distribution of Protein Lengths",
        labels={'Length': 'Protein Length (amino acids)'},
        color_discrete_sequence=['#2ecc71']
    )
    fig.update_layout(
        xaxis_title="Length (amino acids)",
        yaxis_title="Count"
    )
    fig.show()
else:
    print("ℹ️ Column 'Length' not found.")

In [ ]:
# Scatter plot: Length vs p(LLPS)
if 'Length' in df.columns and 'p(LLPS)' in df.columns:
    print("📈 Scatter Plot: Length vs p(LLPS)")
    
    # Color by location if available
    color_col = None
    
    hover_data = [col for col in ['Entry', 'Entry name'] if col in df.columns]
    
    fig = px.scatter(
        df,
        x='Length',
        y='p(LLPS)',
        color=color_col,
        hover_data=hover_data if hover_data else None,
        title="p(LLPS) vs Protein Length"
    )
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.update_layout(
        xaxis_title="Protein Length (amino acids)",
        yaxis_title="p(LLPS)"
    )
    fig.show()
else:
    print("ℹ️ Required columns for scatter plot not found.")

In [23]:
# Location distribution bar chart - Top 20 locations by count
if 'Location Categories' in df.columns:
    print("📍 Protein Count by Subcellular Location (Top 20):")
    
    # Explode and count locations
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    
    # Exclude membrane topology categories (Single-pass, Multi-pass, Lipid-anchor)
    exclude_pattern = 'Single-pass|Multi-pass|Lipid-anchor'
    df_exploded = df_exploded[~df_exploded['Location Categories'].astype(str).str.contains(exclude_pattern, case=False, regex=True)]
    
    location_counts = df_exploded['Location Categories'].value_counts().head(20)
    
    fig = px.bar(
        x=location_counts.index,
        y=location_counts.values,
        title="Protein Count by Subcellular Location - Top 20 (excluding topology)",
        labels={'x': 'Subcellular Location', 'y': 'Number of Proteins'},
        color=location_counts.index,
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, xaxis_tickangle=-45)
    fig.show()
else:
    print("ℹ️ Location column not available.")

📍 Protein Count by Subcellular Location (Top 20):


In [24]:
# Box plot: p(LLPS) by location - Top 20 locations by count
if 'Location Categories' in df.columns and 'p(LLPS)' in df.columns:
    print("📊 p(LLPS) Distribution by Subcellular Location (Top 20):")
    
    # Explode locations for box plot
    df_exploded = df.explode('Location Categories')
    df_exploded = df_exploded[df_exploded['Location Categories'].notna()]
    
    # Exclude membrane topology categories (Single-pass, Multi-pass, Lipid-anchor)
    exclude_pattern = 'Single-pass|Multi-pass|Lipid-anchor'
    df_exploded = df_exploded[~df_exploded['Location Categories'].astype(str).str.contains(exclude_pattern, case=False, regex=True)]
    
    # Get top 20 locations by count
    top_20_locations = df_exploded['Location Categories'].value_counts().head(20).index.tolist()
    df_exploded_top20 = df_exploded[df_exploded['Location Categories'].isin(top_20_locations)]
    
    fig = px.box(
        df_exploded_top20,
        x='Location Categories',
        y='p(LLPS)',
        title="p(LLPS) Distribution by Subcellular Location - Top 20 (excluding topology)",
        color='Location Categories',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, xaxis_tickangle=-45)
    fig.show()
else:
    print("ℹ️ Required columns for box plot not available.")

📊 p(LLPS) Distribution by Subcellular Location (Top 20):


---
## 📝 Notes

This notebook mirrors the data transformations in the Streamlit dashboard (`app.py`) but allows you to:

1. **See intermediate steps**: Each transformation is broken down so you can inspect the data before and after
2. **Modify parameters**: Easily change filter thresholds, search terms, etc.
3. **Add custom analysis**: Extend with your own code cells
4. **Export results**: Save filtered datasets or figures as needed

### Exporting Filtered Data

In [ ]:
# Export filtered data to CSV
# Uncomment and modify the line below to export

# df_filtered.to_csv('filtered_proteins.csv', index=False)
# print("✅ Data exported to filtered_proteins.csv")

---
*Generated from the LLPS Protein Data Explorer project*